# Native AHR v4: task-aware tactics and stronger move improvement

This notebook keeps the native Queen, Worker, Scout and bounded pheromone recurrence, removes JEPA, and gives tactical Workers an explicit goal and target-group mask. The global policy learns game decisions; tactical labels supervise a separate conditioned head. Search uses exact Go transitions, selective adversarial tactical checks, and score/ownership estimates alongside win value.

**Inputs:** tagged definitions from original notebooks, a fixed completed AHR-only v3 checkpoint (`ahr-v3-B-s01`, latest SHA pinned below), and only the train/development records from the frozen tactical curriculum. **Outputs:** reusable tagged definitions and `reports/ahr_native_v4/model_validation.json`. Default Run All is a short validation (normally under two minutes); it does not train a production model or start an arena.

**Resume:** rerun this notebook from the first cell in the `ahr-go` kernel. It has no long-running stage or optimizer checkpoint. Production model/optimizer/RNG/replay checkpoints belong to the trainer notebook under `checkpoints/<run_id>/`. Inference recurrence is fresh for each board; game and training recovery use the trainer/arena manifests. Learned utility and bounded tactical search do not establish full-game optimality.


In [ ]:
from pathlib import Path
import os, json, nbformat
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'notebooks/01_go_rules_and_independent_verifier.ipynb').exists())
_shared = nbformat.read(PROJECT/'experiments/ahr_native_v4/00_shared.ipynb',as_version=4)
for _i,_cell in enumerate(_shared.cells):
    if _cell.cell_type=='code' and 'ahr-export' in _cell.metadata.get('tags',[]):
        exec(compile(_cell.source,f'v4 shared export {_i}','exec'),globals())
_ = v4_load_upstream(PROJECT,globals())


## Native roles with a fast, fresh-board recurrence

The fast path uses the same equations, learned tensors, update order, resets and pheromone bounds as the original recurrence. It does not hash/copy activations to the CPU or retain diagnostic traces at every inference call. Inputs and outputs are checked at public boundaries; validation below compares the fast and original implementations numerically. A recurrent-state continuation must use the original audited implementation: this path intentionally accepts fresh boards only.


In [ ]:
from dataclasses import dataclass, asdict
from collections import defaultdict, Counter
import copy, hashlib, math, time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

V4_GOALS=('capture_one_ply','survive_two_plies','connect_one_ply','force_capture_three_plies')
V4_INITIAL_RUN='ahr-v3-B-s01'
V4_INITIAL_SHA='1cba3e08d002e3882a14ea7230d33ead290e4920810ab6995ba909bc3435647c'

@dataclass(frozen=True)
class V4ModelConfig:
    board_size:int=9
    spatial_channels:int=32
    hidden_dim:int=64
    queen_cycles:int=2
    worker_steps:int=2
    scout_steps:int=2
    dropout:float=.1
    tactical_weight:float=.5
    def __post_init__(self):
        if self.board_size!=9 or self.queen_cycles!=2:
            raise ValueError('V4 protocol uses 9x9 and two Queen cycles')
        GoAHRConfig(**{k:v for k,v in asdict(self).items() if k!='tactical_weight'})
        if isinstance(self.tactical_weight,bool) or not math.isfinite(self.tactical_weight) or self.tactical_weight<0:
            raise ValueError('Tactical loss weight must be finite/nonnegative')

class V4NativeAHR(NativeAHR):
    def forward(self,batch,iterations=None,micro_steps=None,state=None):
        if micro_steps is not None or state is not None:
            raise ValueError('Fast native recurrence requires a fresh board; use original recurrence for continuations')
        x,valid,actions=batch['x'],batch['valid_mask'],batch['action_mask']
        c=self.config; cycles=c.iterations if iterations is None else iterations
        if type(cycles) is not int or not 1<=cycles<=c.max_iterations:
            raise ValueError('Invalid Queen recurrence budget')
        if x.ndim!=3 or x.shape[-1]!=c.input_dim or valid.shape!=x.shape[:2] or actions.shape!=valid.shape or valid.dtype!=torch.bool or actions.dtype!=torch.bool:
            raise ValueError('Invalid native AHR tensor shapes')
        checks=torch.stack((valid.any(-1).all(),actions.any(-1).all(),~(actions&~valid).any(),torch.isfinite(torch.where(valid.unsqueeze(-1),x,torch.zeros_like(x))).all()))
        if not bool(checks.all()):raise ValueError('Invalid native AHR masks or nonfinite visible tokens')
        tokens=self.encoder(torch.where(valid.unsqueeze(-1),x,torch.zeros_like(x)))
        tokens=torch.where(valid.unsqueeze(-1),tokens,torch.zeros_like(tokens))
        pooled=tokens.sum(1)/valid.sum(1,keepdim=True)
        q=torch.tanh(self.initial_queen(pooled)).to(x.dtype)
        w=torch.zeros_like(q);u=torch.zeros_like(q);rho=torch.zeros_like(x[:,:,0])
        for _queen in range(cycles):
            w=torch.tanh(self.reset_worker(torch.cat((q,pooled),-1))).to(x.dtype)
            worker_summary=torch.zeros_like(q)
            for _worker in range(c.worker_steps):
                u=torch.tanh(self.reset_scout(torch.cat((q,w,pooled),-1))).to(x.dtype)
                scout_summary=torch.zeros_like(q)
                for _scout in range(c.scout_steps):
                    scout=self.scout
                    query=scout.query(torch.cat((q,w,u),-1))
                    logits=torch.einsum('bd,bld->bl',query,scout.keys(tokens))/math.sqrt(tokens.shape[-1])
                    weights=(logits+scout.strength*rho).masked_fill(~valid,-torch.inf).float().softmax(-1).to(tokens.dtype)
                    attended=torch.einsum('bl,bld->bd',weights,tokens)
                    context=torch.tanh(scout.context(torch.cat((attended,q,w),-1)))
                    new_u=scout.recurrent(scout.dropout(context),u).to(x.dtype)
                    contexts=[r[:,None,:].expand_as(tokens) for r in (q,w,u)]
                    deposit=c.deposit_scale*self.memory.deposit(torch.cat((tokens,*contexts),-1)).squeeze(-1)
                    deposit=torch.where(valid,deposit,torch.zeros_like(deposit)).to(rho.dtype)
                    rho=pheromone_update(rho,deposit,c,valid)
                    u=new_u;scout_summary=scout_summary+u
                weights=(c.pheromone_strength*rho).masked_fill(~valid,-torch.inf).float().softmax(-1).to(tokens.dtype)
                memory_context=torch.einsum('bl,bld->bd',weights,tokens)
                w=self.worker(w,q,scout_summary/c.scout_steps,memory_context,pooled).to(x.dtype)
                worker_summary=worker_summary+w
            q=self.queen(q,worker_summary/c.worker_steps,memory_context,pooled).to(x.dtype)
        energies={'queen':self.queen.energy(tokens,q),'worker':self.worker.energy(tokens,w),'scout':self.scout.energy(tokens,u),'pheromone':-c.pheromone_strength*rho}
        energies['soft']=sum(energies.values())
        hard=torch.where(actions,torch.zeros_like(rho),torch.full_like(rho,torch.inf))
        return {'policy_logits':-(energies['soft']+hard),'energies':energies,'hard_action_energy':hard,
                'state':{'queen':q,'worker':w,'scout':u,'pheromone':rho},'tokens':tokens,'trace':[],
                'verified_status':'UNKNOWN','fresh_recurrence':True}

class V4AHR(nn.Module):
    def __init__(self,config=None):
        super().__init__()
        self.config=V4ModelConfig(**config) if isinstance(config,dict) else config or V4ModelConfig()
        if not isinstance(self.config,V4ModelConfig):raise TypeError('V4ModelConfig required')
        base=GoAHRConfig(**{k:v for k,v in asdict(self.config).items() if k!='tactical_weight'})
        self.base=GoAHR(base)
        fast=V4NativeAHR(self.base.core.config);fast.load_state_dict(self.base.core.state_dict());self.base.core=fast
        self.goal_embedding=nn.Embedding(len(V4_GOALS),16)
        self.target_encoder=nn.Linear(3,16,bias=False)
        self.tactical_energy=RoleEnergy(self.config.hidden_dim)
    def _tokens(self,batch):
        planes,legal=batch['planes'],batch['legal_mask'];n=self.config.board_size
        if planes.ndim!=4 or tuple(planes.shape[1:])!=(12,n,n) or legal.shape!=(planes.shape[0],n*n+1) or legal.dtype!=torch.bool:
            raise ValueError('Invalid Go feature/mask shape')
        spatial=self.base.spatial(planes).flatten(2).transpose(1,2)
        token=torch.cat((spatial,spatial.mean(1,keepdim=True)+self.base.pass_embedding.reshape(1,1,16)),1)
        return spatial,token,legal
    def forward(self,batch,queen_cycles=None):
        cycles=self.config.queen_cycles if queen_cycles is None else queen_cycles
        spatial,tokens,legal=self._tokens(batch)
        core=self.base.core({'x':tokens,'valid_mask':torch.ones_like(legal),'action_mask':legal},iterations=cycles)
        roles=torch.cat([core['state'][key] for key in ('queen','worker','scout')],-1)
        n=self.config.board_size
        return {**core,'value':torch.tanh(self.base.value_head(roles).squeeze(-1)),
                'score':self.base.score_head(roles).squeeze(-1)*(n*n),
                'ownership':torch.tanh(self.base.ownership_head(torch.cat((spatial,roles[:,None,:].expand(-1,n*n,-1)),-1)).squeeze(-1)),
                'strategic_status':'UNKNOWN'}
    def tactical_forward(self,states,goals,targets):
        if not len(states)==len(goals)==len(targets) or not states:raise ValueError('Nonempty aligned tactical tasks required')
        device=next(self.parameters()).device;batch=encode_go_batch(states,device)
        spatial,tokens,legal=self._tokens(batch);features=np.zeros((len(states),82,3),dtype=np.float32)
        goal_ids=[]
        for i,(state,goal,points) in enumerate(zip(states,goals,targets)):
            if goal not in V4_GOALS or not points or any(type(p)is not int or not 0<=p<81 for p in points):
                raise ValueError('Explicit supported tactical goal and nonempty target mask required')
            goal_ids.append(V4_GOALS.index(goal))
            for p in points:
                features[i,p,0]=1;features[i,p,1]=float(state.board[p]==state.to_play);features[i,p,2]=float(state.board[p]==-state.to_play)
        conditioned=tokens+self.target_encoder(torch.as_tensor(features,device=device))+self.goal_embedding(torch.tensor(goal_ids,device=device))[:,None,:]
        core=self.base.core({'x':conditioned,'valid_mask':torch.ones_like(legal),'action_mask':legal},iterations=self.config.queen_cycles)
        # A separate Worker head prevents local goal labels from directly supervising global policy energies.
        logits=-self.tactical_energy(core['tokens'],core['state']['worker']).masked_fill(~legal,torch.inf)
        return {'policy_logits':logits,'state':core['state'],'goal_ids':goal_ids,'target_features':features}


## Separate global and tactical objectives

Terminal outcomes, scores and ownership retain exact verified labels. UNKNOWN positions may teach legal/search policy but never invented win/loss outcomes. Tactical tasks use their own explicitly conditioned logits; a task without its goal or target mask is rejected. The existing category named `life_death_bounded` is retained only as provenance: its actual predicate is a one-ply capture, not a general life-and-death proof.


In [ ]:
def v4_learning_loss(model,examples):
    if not isinstance(model,V4AHR) or not examples:raise ValueError('V4 model and nonempty examples required')
    tactical=[];global_examples=[]
    for example in examples:
        if any(k in example for k in ('tactical_goal','tactical_targets','tactical_policy_target')):
            if not all(k in example for k in ('tactical_goal','tactical_targets','tactical_policy_target')):
                raise ValueError('Partial tactical conditioning is forbidden')
            if float(example.get('policy_weight',0))!=0:
                raise ValueError('Tactical labels cannot also supervise an unconditioned global policy')
            tactical.append(example)
        else:global_examples.append(example)
    device=next(model.parameters()).device
    if global_examples:supervised,metrics=go_learning_loss(model,global_examples)
    else:
        supervised=next(model.parameters()).sum()*0
        metrics={'loss':0.,'policy':0.,'value':0.,'score':0.,'ownership':0.,'examples':0}
    tactical_loss=supervised*0
    if tactical:
        states=[GoState.from_dict(e['state']) for e in tactical]
        output=model.tactical_forward(states,[e['tactical_goal'] for e in tactical],[e['tactical_targets'] for e in tactical])
        targets=torch.tensor([e['tactical_policy_target'] for e in tactical],device=device,dtype=torch.float32)
        legal=torch.isfinite(output['policy_logits'])
        if targets.shape!=legal.shape or not bool(torch.isfinite(targets).all()) or bool((targets<0).any()) or bool(((targets.sum(-1)-1).abs()>1e-5).any()) or bool((targets[~legal]>0).any()):
            raise ValueError('Tactical target must be a normalized legal distribution')
        logp=F.log_softmax(output['policy_logits'],dim=-1)
        tactical_loss=-(targets*torch.where(targets>0,logp,torch.zeros_like(logp))).sum(-1).mean()
    loss=supervised+model.config.tactical_weight*tactical_loss
    if not bool(torch.isfinite(loss)):raise FloatingPointError('Nonfinite V4 loss')
    metrics.update(loss=float(loss.detach()),tactical=float(tactical_loss.detach()),tactical_examples=len(tactical),global_examples=len(global_examples),queen_cycles=model.config.queen_cycles)
    return loss,metrics

def v4_tactical_curriculum(project,source='experiments/ahr_jepa_v3/data/curriculum.json'):
    path=Path(project)/source;source_sha=hashlib.sha256(path.read_bytes()).hexdigest()
    data=json.loads(path.read_text());result={'train':[],'development':[]}
    for row in data['examples']:
        split=row['split']
        if split not in result:continue  # Lockbox records are never transformed, sampled or evaluated here.
        goal=row['goal']
        if goal not in V4_GOALS:raise ValueError('Unrecognized exact tactical predicate')
        state=GoState.from_dict(row['state']);target=list(row['policy_target'])
        if abs(sum(target)-1)>1e-5 or any(target[a]>0 and a not in legal_actions(state) for a in range(82)):
            raise ValueError('Invalid retained tactical witness policy')
        result[split].append({'state':copy.deepcopy(row['state']),'policy_target':[0.]*82,'policy_weight':0.,
            'tactical_goal':goal,'tactical_targets':list(row['targets']),'tactical_policy_target':target,
            'family_id':row['family_id'],'split':split,'category':row['category'],
            'bounded_objective':copy.deepcopy(row['bounded_objective']),'provenance':copy.deepcopy(row['provenance']),
            'source':'frozen_exact_goal_conditioned_curriculum'})
    train_families={r['family_id'] for r in result['train']};dev_families={r['family_id'] for r in result['development']}
    if not all(result.values()) or train_families&dev_families:raise RuntimeError('Tactical split separation failed')
    result['provenance']={'source':source,'source_sha256':source_sha,'counts':{k:len(v) for k,v in result.items()},
        'goals':list(V4_GOALS),'global_policy_supervision':False,'lockbox_used':False,
        'scope':'Exact bounded predicates with explicit goals/targets; no full-game strength label'}
    return result

def v4_model_initialization(project,device='cuda:1'):
    run_dir=Path(project)/'checkpoints'/V4_INITIAL_RUN;manifest=json.loads((run_dir/'manifest.json').read_text())
    entry=next(e for e in manifest['entries'] if e['file']==manifest['latest']);path=run_dir/entry['file']
    digest=hashlib.sha256(path.read_bytes()).hexdigest()
    if digest!=entry['sha256'] or digest!=V4_INITIAL_SHA:raise RuntimeError('Frozen AHR-only initialization checkpoint differs from declared SHA')
    saved=torch.load(path,map_location='cpu',weights_only=False)
    if saved['config']['use_jepa'] or saved['phase']!='complete':raise RuntimeError('Initialization must be completed AHR-only training')
    config=V4ModelConfig(**saved['model_config']['base_config'])
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(2026100104);model=V4AHR(config)
    weights={k[len('base.'):]:v for k,v in saved['model'].items() if k.startswith('base.')}
    model.base.load_state_dict(weights,strict=True)
    if not all(bool(torch.isfinite(p).all()) for p in model.parameters()):raise RuntimeError('Nonfinite initialization weights')
    model=model.to(device).eval()
    provenance={'kind':'native_ahr_v4','initial_run':V4_INITIAL_RUN,'initial_checkpoint_sha256':digest,
        'selection_rule':'Fixed first AHR-only seed, final checkpoint; no evaluation-based selection',
        'retained_components':'Go spatial encoder, native Queen/Worker/Scout/pheromone, policy/value/score/ownership',
        'new_components':'Goal embedding, target-mask encoder, conditioned tactical Worker energy',
        'external_engine_labels':False,'jepa':False,'config':asdict(config),'initialization_seed':2026100104}
    return model,provenance


## AHR move improvement with selective tactical work

Queen ranks every legal root move using the policy prior and batched successor evaluations. Worker checks selected adversarial continuations, extending unstable local groups within a strict work/time budget. Scout ordering includes exact captures, group escapes and local connection candidates. Every branch uses the real rules; final moves are independently verified. Score, ownership and explicitly labeled local safety estimates provide gradients when win values are similar. Local safety is a ranking heuristic, never a proof that a group lives.

Each completed root continuation is retained if another branch runs out of time. The exported policy combines a sharp preference for the best completed evidence with a temperature-scaled distribution over explored utilities. Exploration belongs to the training actor, not this deterministic evaluation policy. Hard exclusions are legality only; tactical sacrifices remain available.


In [ ]:
@dataclass(frozen=True)
class V4SearchConfig:
    time_budget_s:float=1.
    queen_cycles:int=2
    max_depth:int=4
    max_nodes:int=512
    branch_width:int=8
    tactical_width:int=4
    quiescence_extensions:int=2
    policy_temperature:float=.15
    score_weight:float=.35
    ownership_weight:float=.10
    static_weight:float=.25
    prior_weight:float=.04
    tactical_prior_weight:float=.025
    reserve_fraction:float=.10
    def __post_init__(self):
        if self.queen_cycles!=2 or any(type(x)is not int or x<1 for x in (self.max_depth,self.max_nodes,self.branch_width,self.tactical_width)):
            raise ValueError('Positive integer work budgets and two Queen cycles required')
        if self.max_depth>8 or self.branch_width>82 or type(self.quiescence_extensions)is not int or not 0<=self.quiescence_extensions<=4:
            raise ValueError('Invalid bounded search limits')
        for name in ('time_budget_s','score_weight','ownership_weight','static_weight','prior_weight','tactical_prior_weight'):
            value=getattr(self,name)
            if isinstance(value,bool) or not math.isfinite(value) or value<0:raise ValueError('Invalid '+name)
        if self.score_weight+self.ownership_weight+self.static_weight>1:raise ValueError('Utility weights must total at most one')
        if not math.isfinite(self.policy_temperature) or self.policy_temperature<=0 or not .02<=self.reserve_fraction<=.5:
            raise ValueError('Positive policy temperature and explicit audit reserve required')

class V4SearchCutoff(Exception):pass

def v4_position_key(state):
    return (state.board,state.to_play,state.consecutive_passes,state.history,state.move_number,state.komi)

def v4_true_eye_shape(state,point,color):
    if state.board[point]:return False
    n=state.size;adj=_neighbors(n)[point]
    if not adj or any(state.board[p]!=color for p in adj):return False
    row,col=divmod(point,n);diagonals=[r*n+c for r,c in ((row-1,col-1),(row-1,col+1),(row+1,col-1),(row+1,col+1)) if 0<=r<n and 0<=c<n]
    enemies=sum(state.board[p]==-color for p in diagonals)
    return enemies==0 if len(diagonals)<4 else enemies<=1

def v4_static_score(state,perspective=None):
    # A local safety estimate, not territory/life certification. Empty open regions receive no assumed owner.
    perspective=state.to_play if perspective is None else perspective
    score=float(sum(state.board)-state.komi)
    for stones,libs in groups_and_liberties(state.board,state.size):
        color=state.board[next(iter(stones))];size=len(stones)
        safety=.10*size*min(len(libs),4)-1.5*size*int(len(libs)==1)-.25*size*int(len(libs)==2)
        eyes=sum(v4_true_eye_shape(state,p,color) for p in libs)
        safety+=.35*size*min(eyes,2)
        score+=color*safety
    return perspective*score

def v4_tactical_tasks(state,maximum=4):
    tasks=[]
    for stones,libs in groups_and_liberties(state.board,state.size):
        color=state.board[next(iter(stones))]
        if len(libs)==1:
            goal='survive_two_plies' if color==state.to_play else 'capture_one_ply'
            tasks.append((len(stones)+2,goal,sorted(stones)))
        elif color==-state.to_play and len(libs)==2:
            tasks.append((.25*len(stones),'force_capture_three_plies',sorted(stones)))
    tasks.sort(key=lambda x:(-x[0],x[1],x[2]))
    return tasks[:maximum]

def v4_move_features(state,action,child,groups=None):
    if action==state.pass_action:
        return {'captures':0,'saved_stones':0,'self_atari':0,'eye_fill':0,'connections':0,'priority':-.15}
    groups=groups_and_liberties(state.board,state.size) if groups is None else groups
    color=state.to_play;captured=sum(x==-color for x in state.board)-sum(x==-color for x in child.board)
    saved=0
    for stones,libs in groups:
        anchor=next(iter(stones))
        if state.board[anchor]==color and len(libs)==1 and child.board[anchor]==color:
            _,after=group_and_liberties(child.board,child.size,anchor)
            if len(after)>=2:saved+=len(stones)
    placed,libs=group_and_liberties(child.board,child.size,action)
    self_atari=len(placed) if len(libs)==1 else 0
    eye_fill=int(v4_true_eye_shape(state,action,color) and captured==0 and saved==0)
    adjacent=set(_neighbors(state.size)[action]);connections=sum(bool(set(stones)&adjacent) for stones,_ in groups if state.board[next(iter(stones))]==color)
    priority=1.5*captured+.8*saved-.8*self_atari-2.5*eye_fill+.20*max(0,connections-1)
    return {'captures':captured,'saved_stones':saved,'self_atari':self_atari,'eye_fill':eye_fill,'connections':connections,'priority':priority}

def v4_choose_move(model,state,search_config=None):
    config=V4SearchConfig(**search_config) if isinstance(search_config,dict) else search_config or V4SearchConfig()
    if not isinstance(model,V4AHR) or not isinstance(state,GoState) or state.size!=9 or model.training:
        raise ValueError('Evaluation-mode V4AHR and exact 9x9 GoState required')
    start=time.monotonic();deadline=start+config.time_budget_s*(1-config.reserve_fraction)
    device=next(model.parameters()).device;cache={};groups_cache={};feature_cache={}
    stats={'nodes':0,'model_calls':0,'model_positions':0,'queen_iterations':0,'worker_transitions':0,
           'scout_candidates':0,'alpha_beta_prunes':0,'tactical_model_calls':0,'selective_extensions':0}
    def check():
        if time.monotonic()>=deadline:raise V4SearchCutoff('time_budget')
        if stats['nodes']>=config.max_nodes:raise V4SearchCutoff('node_budget')
    def exact_child(position,action):
        check();child=play(position,action);stats['worker_transitions']+=1;return child
    def groups(position):
        key=v4_position_key(position)
        if key not in groups_cache:groups_cache[key]=groups_and_liberties(position.board,position.size)
        return groups_cache[key]
    def features(position,action,child):
        key=(v4_position_key(position),action)
        if key not in feature_cache:feature_cache[key]=v4_move_features(position,action,child,groups(position))
        return feature_cache[key]
    def evaluate(positions):
        pending=[];seen=set()
        for position in positions:
            key=v4_position_key(position)
            if key not in cache and key not in seen and not position.terminal:pending.append((key,position));seen.add(key)
        if not pending:return
        check()
        with torch.inference_mode():out=model(encode_go_batch([s for _,s in pending],device),queen_cycles=config.queen_cycles)
        values=out['value'].detach().cpu().tolist();scores=out['score'].detach().cpu().tolist()
        owners=out['ownership'].detach().mean(-1).cpu().tolist();logits=out['policy_logits'].detach().cpu().numpy()
        if not all(math.isfinite(x) for x in values+scores+owners) or np.isnan(logits).any():raise FloatingPointError('Nonfinite AHR inference')
        for i,(key,position) in enumerate(pending):
            policy=np.exp(logits[i]-np.max(logits[i]));policy/=policy.sum()
            value_weight=1-config.score_weight-config.ownership_weight-config.static_weight
            utility=(value_weight*values[i]+config.score_weight*math.tanh(scores[i]/20)+
                     config.ownership_weight*owners[i]+config.static_weight*math.tanh(v4_static_score(position)/20))
            cache[key]={'utility':float(utility),'value':float(values[i]),'score':float(scores[i]),'prior':policy}
        stats['model_calls']+=1;stats['model_positions']+=len(pending)
    def utility(position):
        if position.terminal:
            relative=position.to_play*area_score(position)
            return float((relative>0)-(relative<0))
        evaluate([position]);return cache[v4_position_key(position)]['utility']
    def ordered_children(position,width,forcing_only=False):
        check();evaluate([position]);prior=cache[v4_position_key(position)]['prior'];legal=legal_actions(position)
        forcing=set()
        for stones,libs in groups(position):
            if len(libs)<=2:forcing.update(libs)
        forcing.intersection_update(legal)
        ranked=sorted(legal,key=lambda a:(a not in forcing,-prior[a],a))
        if forcing_only:ranked=[a for a in ranked if a in forcing]
        else:ranked=ranked[:max(width,len(forcing))]
        children=[]
        for action in ranked:
            child=exact_child(position,action);feat=features(position,action,child)
            children.append((action,child,feat))
        children.sort(key=lambda item:(-item[2]['priority'],-prior[item[0]],item[0]))
        children=children[:width];stats['scout_candidates']+=len(children)
        return children
    def visit(position,depth,extensions,alpha=-float('inf'),beta=float('inf'),local_limit=None):
        check()
        if local_limit is not None and stats['nodes']>=local_limit:raise V4SearchCutoff('branch_work_budget')
        stats['nodes']+=1
        if position.terminal:return utility(position)
        if position.consecutive_passes==1:
            terminal=exact_child(position,position.pass_action)
            if position.to_play*area_score(terminal)>0:return 1.
        evaluate([position])
        unstable=any(len(libs)==1 for _,libs in groups(position))
        if depth<=0 and not (extensions and unstable):return utility(position)
        forcing_only=depth<=0
        if forcing_only:stats['selective_extensions']+=1
        children=ordered_children(position,config.tactical_width,forcing_only)
        if not children:return utility(position)
        evaluate([c for _,c,_ in children])
        best=-float('inf')
        for action,child,feat in children:
            value=-visit(child,max(0,depth-1),extensions-int(forcing_only),-beta,-alpha,local_limit)
            best=max(best,value);alpha=max(alpha,value)
            if alpha>=beta:stats['alpha_beta_prunes']+=1;break
        return best
    base={'strategic_status':'UNKNOWN','verified_legal':True,'policy_target_kind':'sharp_completed_ahr_evidence',
          'jepa':False,'utility_scope':'Learned score/ownership plus bounded AHR continuations; not a full-game proof'}
    if state.terminal:
        return {**base,'action':None,'policy':[0.]*82,'policy_target':[0.]*82,'root_value':utility(state),'status':'TERMINAL',
                'completed_depth':0,'elapsed_s':time.monotonic()-start,'deadline_missed':False,**stats}
    legal=list(legal_actions(state));chosen=state.pass_action;reason='zero_budget';root_values={};depths={};root_prior=np.zeros(82);root_prior[legal]=1/len(legal)
    root_children={};root_features={};completed_root_checks=0;root_neural_value=None
    if state.consecutive_passes==1 and config.time_budget_s>0:
        terminal=play(state,state.pass_action)
        if not verify_go_emission(state,state.pass_action,terminal) or area_score(terminal)!=arena_reference_score(terminal):raise RuntimeError('Independent terminal-pass audit disagrees')
        if state.to_play*area_score(terminal)>0:
            p=[0.]*82;p[81]=1.;elapsed=time.monotonic()-start
            return {**base,'action':81,'policy':p,'policy_target':p,'root_value':1.,'status':'VERIFIED_TERMINAL_WIN',
                    'completed_depth':1,'elapsed_s':elapsed,'deadline_missed':elapsed>config.time_budget_s,'reason':'verified_terminal_win',**stats}
    if config.time_budget_s>0:
        try:
            evaluate([state]);root=cache[v4_position_key(state)];root_prior=root['prior'];root_neural_value=root['value']
            chosen=max(legal,key=lambda a:(root_prior[a],-a));reason='native_prior'
            # Scout observes every legal root successor, eliminating the former top-eight blind spot.
            for action in legal:
                child=exact_child(state,action);root_children[action]=child;root_features[action]=features(state,action,child)
            evaluate(list(root_children.values()))
            for action,child in root_children.items():root_values[action]=-utility(child);depths[action]=1
            tasks=v4_tactical_tasks(state,maximum=2)
            tactical_bonus=np.zeros(82)
            if tasks and config.tactical_prior_weight:
                check()
                with torch.inference_mode():task_output=model.tactical_forward([state]*len(tasks),[t[1] for t in tasks],[t[2] for t in tasks])
                task_probs=task_output['policy_logits'].softmax(-1).detach().cpu().numpy()
                tactical_bonus=task_probs.max(0);stats['tactical_model_calls']+=1
            def ranking(action):
                return (root_values[action]+config.prior_weight*math.log(max(root_prior[action],1e-8))+
                        .06*math.tanh(root_features[action]['priority']/4)+config.tactical_prior_weight*tactical_bonus[action])
            candidates=sorted(root_values,key=lambda a:(-ranking(a),-root_prior[a],a))[:config.branch_width]
            urgent=sorted([a for a in root_values if root_features[a]['captures'] or root_features[a]['saved_stones']],key=lambda a:(-root_features[a]['priority'],-ranking(a),a))[:config.tactical_width]
            candidates=list(dict.fromkeys(urgent+candidates))
            chosen=max(root_values,key=lambda a:(ranking(a),root_prior[a],-a));reason='root_successor_evidence'
            for depth in range(2,config.max_depth+1):
                check();stats['queen_iterations']+=1
                frontier=sorted(candidates,key=lambda a:(-ranking(a),depths[a],a))
                for action in frontier:
                    check();local_limit=min(config.max_nodes,stats['nodes']+max(24,config.max_nodes//max(2*len(candidates),1)))
                    try:value=-visit(root_children[action],depth-1,config.quiescence_extensions,local_limit=local_limit)
                    except V4SearchCutoff as cutoff:
                        if str(cutoff)=='branch_work_budget':continue
                        raise
                    root_values[action]=value;depths[action]=depth;completed_root_checks+=1
                    chosen=max(root_values,key=lambda a:(ranking(a),root_prior[a],-a))
                    reason='completed_selective_ahr_evidence'
        except V4SearchCutoff as cutoff:reason=str(cutoff)
    # A completed estimate is kept for every explored root move; a cutoff never becomes a proof of loss.
    policy=np.zeros(82,dtype=float)
    if root_values:
        moves=sorted(root_values);values=np.array([root_values[a] for a in moves])
        logits=(values-values.max())/config.policy_temperature+.10*np.log(np.maximum(root_prior[moves],1e-8))
        soft=np.exp(logits-logits.max());soft/=soft.sum()
        evidence_weight=.65 if completed_root_checks else .5
        policy[moves]=(1-evidence_weight)*soft;policy[chosen]+=evidence_weight
        root_value=float(root_values[chosen])
    else:
        policy[legal]=root_prior[legal];policy/=policy.sum();root_value=root_neural_value
    child=play(state,chosen)
    if not verify_go_emission(state,chosen,child):raise RuntimeError('Independent final emission audit rejected move')
    elapsed=time.monotonic()-start;missed=elapsed>config.time_budget_s
    return {**base,'action':int(chosen),'policy':policy.tolist(),'policy_target':policy.tolist(),'root_value':root_value,
        'root_neural_value':root_neural_value,'status':'UNKNOWN','reason':reason,'completed_depth':depths.get(chosen,0),
        'root_depths':{str(k):v for k,v in depths.items()},'completed_root_checks':completed_root_checks,
        'root_candidates':list(root_values),'root_tactical_candidates':[a for a,f in root_features.items() if f['captures'] or f['saved_stones']],
        'fallback_pass':not bool(root_values) and chosen==81,'elapsed_s':elapsed,'deadline_missed':missed,**stats}


## Bounded validation

The checks compare original and fast native outputs and base gradients, verify task-conditioning and separation of policy gradients, check legal masks and exact terminal passing, and exercise the full one-second decision path on opening/tactical positions. These tests establish implementation properties, not playing strength. Production training and GNU matches run in the later notebooks.


In [ ]:
def v4_validate_model(project,device='cpu'):
    v4_seed(2026100104)
    curriculum=v4_tactical_curriculum(project)
    model,provenance=v4_model_initialization(project,device)
    states=[initial_state(9,7.5),GoState.from_dict(curriculum['train'][0]['state'])]
    original=GoAHR(model.base.config).to(device).eval();original.load_state_dict(model.base.state_dict())
    batch=encode_go_batch(states,device)
    model.eval();original.eval()
    fast=model(batch);slow=original(batch,queen_cycles=2)
    errors={key:float((fast[key]-slow[key]).abs().max().detach()) for key in ('value','score','ownership')}
    legal=batch['legal_mask'];errors['policy_logits']=float((fast['policy_logits'][legal]-slow['policy_logits'][legal]).abs().max().detach())
    assert all(v<3e-5 for v in errors.values()),errors
    for key in ('queen','worker','scout','pheromone'):
        assert torch.allclose(fast['state'][key],slow['state'][key],atol=3e-6,rtol=3e-6),key
    model.zero_grad(set_to_none=True);original.zero_grad(set_to_none=True)
    fast_loss=fast['value'].sum()+fast['score'].sum()/81+fast['ownership'].mean()+fast['policy_logits'][legal].mean()
    slow_loss=slow['value'].sum()+slow['score'].sum()/81+slow['ownership'].mean()+slow['policy_logits'][legal].mean()
    fast_loss.backward();slow_loss.backward()
    gradient_error=0.
    for name,param in original.named_parameters():
        other=dict(model.base.named_parameters())[name]
        if param.grad is not None:
            assert other.grad is not None,name
            gradient_error=max(gradient_error,float((param.grad-other.grad).abs().max()))
            assert torch.allclose(param.grad,other.grad,atol=3e-5,rtol=3e-5),name
    model.zero_grad(set_to_none=True)
    tasks=curriculum['train'][:8]
    model.train();loss,metrics=v4_learning_loss(model,tasks);loss.backward()
    assert model.goal_embedding.weight.grad is not None and model.goal_embedding.weight.grad.abs().sum()>0
    assert model.target_encoder.weight.grad is not None and model.target_encoder.weight.grad.abs().sum()>0
    assert model.tactical_energy.network[0].weight.grad is not None
    forbidden=[]
    for name,param in model.base.named_parameters():
        if any(part in name for part in ('queen.energy','worker.energy','scout.energy','value_head','score_head','ownership_head')):
            assert param.grad is None or bool((param.grad==0).all()),name
            forbidden.append(name)
    model.eval();model.zero_grad(set_to_none=True)
    row=tasks[0];position=GoState.from_dict(row['state'])
    alternate=next(p for p in range(81) if p not in row['tactical_targets'])
    with torch.inference_mode():
        a=model.tactical_forward([position]*2,[row['tactical_goal']]*2,[row['tactical_targets'],[alternate]])['policy_logits']
    finite=torch.isfinite(a[0])&torch.isfinite(a[1]);target_effect=float((a[0,finite]-a[1,finite]).abs().max())
    assert target_effect>1e-7,target_effect
    rejected=False
    try:v4_learning_loss(model,[{**row,'policy_weight':1.}])
    except ValueError:rejected=True
    assert rejected
    malformed={k:v.clone() for k,v in batch.items()}
    malformed['legal_mask'].zero_()
    rejected=False
    try:model(malformed)
    except ValueError:rejected=True
    assert rejected
    # Warm kernels before timing the public one-second path.
    with torch.inference_mode():model(encode_go_batch(states,device))
    board=[0]*81;board[40]=-1
    for point in (31,39,49):board[point]=1
    capture=GoState(9,tuple(board),to_play=1)
    decisions=[]
    for name,position in [('opening',states[0]),('one_ply_capture',capture),('training_tactical_position',states[1])]:
        result=v4_choose_move(model,position,V4SearchConfig())
        assert result['action'] in legal_actions(position)
        assert verify_go_emission(position,result['action'],play(position,result['action']))
        assert abs(sum(result['policy'])-1)<1e-8 and all(p==0 for i,p in enumerate(result['policy']) if i not in legal_actions(position))
        if result['completed_depth']:assert result['policy'][result['action']]>=.5
        assert not result['deadline_missed'],(name,result['elapsed_s'])
        if name=='one_ply_capture':assert 41 in result['root_tactical_candidates'] and result['action']==41
        decisions.append({'case':name,**{k:result[k] for k in ('action','completed_depth','nodes','model_positions','elapsed_s','deadline_missed','completed_root_checks')},'selected_policy_probability':result['policy'][result['action']]})
    passed=play(initial_state(9,7.5),81);terminal=v4_choose_move(model,passed,V4SearchConfig())
    assert terminal['action']==81 and terminal['status']=='VERIFIED_TERMINAL_WIN'
    assert terminal['policy'][81]==1. and play(passed,81).terminal
    zero=v4_choose_move(model,initial_state(9,7.5),V4SearchConfig(time_budget_s=0))
    assert zero['status']=='UNKNOWN' and zero['action'] in legal_actions(initial_state(9,7.5))
    return {'status':'PASS','device':device,'model_config':asdict(model.config),'parameters':sum(p.numel() for p in model.parameters()),
        'initialization':provenance,'native_forward_max_error':errors,'native_gradient_max_error':gradient_error,
        'tactical_target_effect':target_effect,'tactical_policy_separation_checked_parameters':len(forbidden),
        'tactical_conditioning_loss':metrics['tactical'],'curriculum':curriculum['provenance'],'decisions':decisions,
        'exact_terminal_pass':'PASS','zero_budget_unknown':'PASS','production_training_started':False,
        'strength_claim':'NOT_ESTABLISHED','model_notebook_sha256':v4_code_sha(Path(project)/'experiments/ahr_native_v4/01_native_ahr_and_tactics.ipynb')}


In [ ]:
validation=v4_validate_model(PROJECT,device=os.environ.get('AHRV4_MODEL_VALIDATION_DEVICE','cpu'))
_report=v4_report_path(PROJECT,'model_validation.json')
atomic_json(_report,validation)
from IPython.display import display,Markdown
_rows='\n'.join(f"| {r['case']} | {r['action']} | {r['completed_depth']} | {r['nodes']} | {r['elapsed_s']:.3f} | {r['selected_policy_probability']:.3f} |" for r in validation['decisions'])
display(Markdown(f"**Validation: {validation['status']}** — {validation['device']}; {validation['parameters']:,} parameters; no JEPA.\n\n| Position | Move index | Selected depth | Worker nodes | Seconds | Target probability |\n|---|---:|---:|---:|---:|---:|\n{_rows}\n\nNative numerical equivalence, goal/target conditioning, separate tactical gradients, exact terminal passing and legal final moves passed. Playing strength remains unmeasured."))
